# LuFlow Network Anomaly Detection — Modelling

Implements 4 anomaly-detection models with strict temporal splits (train=2020, val=2021, test=2022).  
No data from val/test years leaks into preprocessing fit steps.

| Model | Type | Training set |
|-------|------|-------------|
| Isolation Forest | Semi-supervised | All 2020 flows |
| Autoencoder | Unsupervised | Benign-only 2020 flows |
| Logistic Regression | Supervised baseline | All 2020 flows |
| Random Forest | Supervised strong baseline | All 2020 flows |

## Phase 1 — Setup & Data Loading

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100

# ── Paths ──────────────────────────────────────────────────────────────────────
_here = Path().resolve()
REPO_ROOT  = _here.parent if _here.name == "notebooks" else _here
DATA_DIR   = REPO_ROOT / "data" / "LuFlow-dataset"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Global flags ───────────────────────────────────────────────────────────────
SAMPLE_MODE          = True   # set False to load full dataset
SAMPLE_DAYS_PER_YEAR = 7
CHUNK_SIZE           = 200_000
SEED                 = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}  device={device}")
print(f"DATA    : {DATA_DIR}  (exists={DATA_DIR.exists()})")

In [ ]:
# ── Schema definitions ─────────────────────────────────────────────────────────
NUMERIC_COLS = [
    "avg_ipt", "bytes_in", "bytes_out",
    "entropy", "num_pkts_out", "num_pkts_in",
    "proto", "time_end", "time_start",
    "total_entropy", "duration",
    "src_port", "dest_port",
]
LABEL_COL = "label"

# 11 modelling features (no raw IPs or timestamps)
FEATURE_COLS = [
    "avg_ipt", "bytes_in", "bytes_out", "entropy",
    "num_pkts_in", "num_pkts_out", "total_entropy",
    "duration", "src_port", "dest_port", "proto",
]

# Heavy right-skew → log1p per exploration findings
LOG1P_COLS = ["bytes_in", "bytes_out", "num_pkts_in", "num_pkts_out", "duration", "avg_ipt"]


def discover_csv_files(data_dir: Path) -> pd.DataFrame:
    """Recursively find all CSVs and parse date metadata from path structure.
    Expected layout: <data_dir>/<YYYY>/<MM>/<YYYY.MM.DD>/<YYYY.MM.DD>.csv
    """
    records = []
    for csv_path in sorted(data_dir.rglob("*.csv")):
        rel   = csv_path.relative_to(data_dir)
        parts = rel.parts
        if len(parts) < 4:
            continue
        year_str, _month_str, day_dir = parts[0], parts[1], parts[2]
        try:
            parsed_date = pd.Timestamp(day_dir.replace(".", "-"))
        except Exception:
            parsed_date = pd.NaT
        records.append({
            "path"        : str(csv_path),
            "year"        : int(year_str),
            "month"       : int(_month_str),
            "date"        : parsed_date,
            "file_size_mb": round(csv_path.stat().st_size / 1e6, 3),
        })
    return pd.DataFrame(records)


def normalise_df(raw: pd.DataFrame, file_date: pd.Timestamp) -> pd.DataFrame:
    """Apply schema normalisation to a raw (all-string) DataFrame."""
    df = raw.copy()
    df = df.replace("", np.nan)
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in ["time_start", "time_end"]:
        if col in df.columns:
            df.loc[df[col] < 1e12, col] = np.nan
    if LABEL_COL in df.columns:
        df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
    df["date"] = file_date
    return df


file_df = discover_csv_files(DATA_DIR)
print(f"Total CSV files : {len(file_df):,}")
print(f"Date range      : {file_df['date'].min().date()} → {file_df['date'].max().date()}")
file_df.groupby("year").size().rename("files").to_frame()

In [ ]:
if SAMPLE_MODE:
    files_to_load = (
        file_df
        .groupby("year", group_keys=False)
        .apply(lambda g: g.nsmallest(SAMPLE_DAYS_PER_YEAR, "date"))
        .reset_index(drop=True)
    )
    print(f"SAMPLE_MODE: loading {len(files_to_load)} files "
          f"({SAMPLE_DAYS_PER_YEAR} days/year) ...")
else:
    files_to_load = file_df.copy()
    print(f"FULL_MODE: loading {len(files_to_load)} files ...")

chunks: list[pd.DataFrame] = []
for _, row in files_to_load.iterrows():
    try:
        if SAMPLE_MODE:
            raw = pd.read_csv(row["path"], dtype=str, low_memory=False)
            chunks.append(normalise_df(raw, row["date"]))
        else:
            for chunk in pd.read_csv(
                row["path"], dtype=str, chunksize=CHUNK_SIZE, low_memory=False
            ):
                chunks.append(normalise_df(chunk, row["date"]))
    except Exception as exc:
        print(f"  WARN {row['path']}: {exc}")

df = pd.concat(chunks, ignore_index=True)
df["year"] = df["date"].dt.year

# Binary target: benign=0, malicious/outlier=1
df["target"] = (df[LABEL_COL] != "benign").astype(int)

print(f"\nLoaded        : {df.shape}")
print(f"Memory        : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nLabel counts  :\n{df[LABEL_COL].value_counts()}")
print(f"\nBinary target :\n{df['target'].value_counts()}")
print(f"\nYear counts   :\n{df['year'].value_counts().sort_index()}")

## Phase 2 — Feature Engineering

- `log1p` on: `bytes_in`, `bytes_out`, `num_pkts_in`, `num_pkts_out`, `duration`, `avg_ipt` (heavy right-skew per exploration)
- Median imputation for `src_port`, `dest_port` (NaN from empty strings in raw CSVs)
- `StandardScaler` fitted **on train only** to prevent leakage

In [ ]:
avail_features = [c for c in FEATURE_COLS if c in df.columns]
avail_log1p    = [c for c in LOG1P_COLS if c in avail_features]

model_df = df[avail_features + ["year", "target", LABEL_COL]].copy()

# log1p on heavy right-skewed features (clip to 0 first to handle rare negatives)
for col in avail_log1p:
    model_df[col] = np.log1p(model_df[col].clip(lower=0))

print(f"Features  ({len(avail_features)}): {avail_features}")
print(f"log1p applied to : {avail_log1p}")
print(f"Shape            : {model_df.shape}")
model_df[avail_features].describe().T[["mean", "std", "min", "max"]].round(3)

## Phase 3 — Temporal Split

- **Train**: year 2020 (~35% anomalous)
- **Val**: year 2021 (~15% anomalous) — used for threshold tuning
- **Test**: year 2022 (~52% anomalous) — held-out evaluation only

No random shuffle across dates; temporal ordering is preserved.

In [ ]:
train_df = model_df[model_df["year"] == 2020].copy()
val_df   = model_df[model_df["year"] == 2021].copy()
test_df  = model_df[model_df["year"] == 2022].copy()

print(f"Train (2020): {len(train_df):,} rows  anomaly_rate={train_df['target'].mean():.3f}")
print(f"Val   (2021): {len(val_df):,} rows   anomaly_rate={val_df['target'].mean():.3f}")
print(f"Test  (2022): {len(test_df):,} rows  anomaly_rate={test_df['target'].mean():.3f}")

# Median imputation — compute medians on train only, apply to all splits
train_medians = train_df[avail_features].median()
for col in avail_features:
    train_df[col] = train_df[col].fillna(train_medians[col])
    val_df[col]   = val_df[col].fillna(train_medians[col])
    test_df[col]  = test_df[col].fillna(train_medians[col])

# StandardScaler fit on train only — prevents leakage
scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[avail_features])
X_val   = scaler.transform(val_df[avail_features])
X_test  = scaler.transform(test_df[avail_features])

y_train = train_df["target"].values
y_val   = val_df["target"].values
y_test  = test_df["target"].values

label_train = train_df[LABEL_COL].values
label_val   = val_df[LABEL_COL].values
label_test  = test_df[LABEL_COL].values

train_anomaly_rate = float(y_train.mean())
print(f"\nContamination (train anomaly rate) : {train_anomaly_rate:.4f}")
print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")

## Phase 4 — Isolation Forest

Semi-supervised: trained on all 2020 flows.  
`contamination` parameter set from observed training anomaly prevalence.  
Threshold tuned on val set to maximise F1.

In [ ]:
# IsolationForest — semi-supervised, trained on all 2020 data
# contamination derived from observed training label prevalence
print(f"Training IsolationForest  contamination={train_anomaly_rate:.4f} ...")
iso_forest = IsolationForest(
    contamination=train_anomaly_rate,
    random_state=SEED,
    n_jobs=-1,
)
iso_forest.fit(X_train)

# Anomaly score: negate decision_function so higher score = more anomalous
if_score_val  = -iso_forest.decision_function(X_val)
if_score_test = -iso_forest.decision_function(X_test)

print("IsolationForest trained.")
print(f"Val  score range : [{if_score_val.min():.4f}, {if_score_val.max():.4f}]")
print(f"Test score range : [{if_score_test.min():.4f}, {if_score_test.max():.4f}]")

In [ ]:
# ── Shared evaluation helpers (used by all models below) ──────────────────────
results_log: list[dict] = []   # accumulates test metrics across all models


def sweep_threshold_f1(y_true: np.ndarray, y_score: np.ndarray, n_points: int = 300) -> tuple[float, float]:
    """Find the threshold that maximises F1 on y_true."""
    lo, hi = np.percentile(y_score, 1), np.percentile(y_score, 99)
    best_f1, best_t = 0.0, lo
    for t in np.linspace(lo, hi, n_points):
        f1 = f1_score(y_true, (y_score >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1


def evaluate_model(name: str, y_true: np.ndarray, y_score: np.ndarray, threshold: float) -> dict:
    """Return ROC-AUC, F1@threshold, and Precision@Recall>=0.9."""
    y_pred = (y_score >= threshold).astype(int)
    auc    = roc_auc_score(y_true, y_score)
    f1     = f1_score(y_true, y_pred, zero_division=0)
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    mask     = rec[:-1] >= 0.9
    p_at_r90 = float(prec[:-1][mask].max()) if mask.any() else 0.0
    return {
        "model"    : name,
        "roc_auc"  : round(float(auc),  4),
        "f1"       : round(float(f1),   4),
        "p_at_r90" : round(p_at_r90,    4),
    }


# ── Isolation Forest — val threshold sweep → test evaluation ──────────────────
if_thresh, if_val_f1 = sweep_threshold_f1(y_val, if_score_val)
print(f"IF best val threshold : {if_thresh:.6f}  val_F1={if_val_f1:.4f}")

if_metrics = evaluate_model("IsolationForest", y_test, if_score_test, if_thresh)
results_log.append(if_metrics)
print(f"Test metrics          : {if_metrics}")

print("\nPer-label detection rate on test:")
for cls in sorted(set(label_test)):
    mask = label_test == cls
    rate = (if_score_test[mask] >= if_thresh).mean()
    print(f"  {cls:12s}: {mask.sum():6,} samples  detected={rate:.3f}")

## Phase 5 — Autoencoder

Architecture: `input_dim → 64 → 32 → 16 → 32 → 64 → input_dim` (ReLU, bottleneck=16)  
Trained on **benign-only** 2020 flows; reconstruction MSE is the anomaly score.  
Early stopping on benign-only val reconstruction loss.

In [ ]:
# ── Autoencoder architecture ───────────────────────────────────────────────────
class TabularAE(nn.Module):
    """Tabular autoencoder: input_dim → 64 → 32 → 16 → 32 → 64 → input_dim."""

    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32),        nn.ReLU(),
            nn.Linear(32, 16),        nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, input_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


# Benign-only training subset
benign_mask_train = label_train == "benign"
X_train_benign    = X_train[benign_mask_train]
print(f"Benign train samples : {X_train_benign.shape[0]:,} ({benign_mask_train.mean():.1%} of train)")

input_dim  = X_train.shape[1]
ae_model   = TabularAE(input_dim).to(device)
ae_optim   = torch.optim.Adam(ae_model.parameters(), lr=1e-3)
ae_loss_fn = nn.MSELoss()

# Torch tensors
X_train_benign_t = torch.tensor(X_train_benign, dtype=torch.float32)
X_val_t          = torch.tensor(X_val,          dtype=torch.float32)
X_test_t         = torch.tensor(X_test,         dtype=torch.float32)

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train_benign_t),
    batch_size=256,
    shuffle=True,
)

print(f"Architecture : {input_dim} → 64 → 32 → 16 → 32 → 64 → {input_dim}")
print(f"Parameters   : {sum(p.numel() for p in ae_model.parameters()):,}")

In [ ]:
AE_MAX_EPOCHS = 100
AE_PATIENCE   = 10

train_losses: list[float] = []
val_losses:   list[float] = []
best_val_loss = float("inf")
patience_ctr  = 0
best_state    = None

# Boolean index for benign-only val early-stopping
val_benign_idx = torch.tensor(label_val == "benign", dtype=torch.bool)
if val_benign_idx.sum() == 0:
    val_benign_idx = torch.ones(len(X_val_t), dtype=torch.bool)
X_val_benign_t = X_val_t[val_benign_idx]

for epoch in range(1, AE_MAX_EPOCHS + 1):
    # ── train pass ────────────────────────────────────────────────────────────
    ae_model.train()
    epoch_loss = 0.0
    for (batch_x,) in train_loader:
        batch_x = batch_x.to(device)
        recon   = ae_model(batch_x)
        loss    = ae_loss_fn(recon, batch_x)
        ae_optim.zero_grad()
        loss.backward()
        ae_optim.step()
        epoch_loss += loss.item() * len(batch_x)
    epoch_loss /= len(X_train_benign_t)

    # ── validation pass (benign-only) ─────────────────────────────────────────
    ae_model.eval()
    with torch.no_grad():
        xvb      = X_val_benign_t.to(device)
        val_loss = ae_loss_fn(ae_model(xvb), xvb).item()

    train_losses.append(epoch_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.cpu().clone() for k, v in ae_model.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= AE_PATIENCE:
            print(f"Early stopping at epoch {epoch}  (best val loss: {best_val_loss:.6f})")
            break

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  train={epoch_loss:.6f}  val={val_loss:.6f}")

if best_state is not None:
    ae_model.load_state_dict(best_state)

print(f"\nBest val loss : {best_val_loss:.6f}")
print(f"Epochs run    : {len(train_losses)}")

# ── Loss curve ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="train (benign-only)")
ax.plot(val_losses,   label="val  (benign-only)")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Autoencoder Reconstruction Loss")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ae_loss_curve.png", bbox_inches="tight")
plt.show()

monotone = all(a >= b for a, b in zip(train_losses, train_losses[1:]))
print(f"Train loss monotonically non-increasing: {monotone}")

In [ ]:
# ── Autoencoder — per-sample MSE as anomaly score ─────────────────────────────
ae_model.eval()
with torch.no_grad():
    rec_val      = ae_model(X_val_t.to(device)).cpu()
    ae_mse_val   = ((rec_val  - X_val_t) ** 2).mean(dim=1).numpy()
    rec_test     = ae_model(X_test_t.to(device)).cpu()
    ae_mse_test  = ((rec_test - X_test_t) ** 2).mean(dim=1).numpy()

ae_thresh, ae_val_f1 = sweep_threshold_f1(y_val, ae_mse_val)
print(f"AE best val threshold : {ae_thresh:.6f}  val_F1={ae_val_f1:.4f}")

ae_metrics = evaluate_model("Autoencoder", y_test, ae_mse_test, ae_thresh)
results_log.append(ae_metrics)
print(f"Test metrics          : {ae_metrics}")

print("\nPer-label detection rate on test:")
for cls in sorted(set(label_test)):
    mask = label_test == cls
    rate = (ae_mse_test[mask] >= ae_thresh).mean()
    print(f"  {cls:12s}: {mask.sum():6,} samples  detected={rate:.3f}")

## Phase 6 — Supervised Models

Logistic Regression and Random Forest with `class_weight="balanced"` to handle imbalance.

In [ ]:
# ── Logistic Regression ────────────────────────────────────────────────────────
print("Training LogisticRegression ...")
lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1,
)
lr_model.fit(X_train, y_train)

lr_score_val  = lr_model.predict_proba(X_val)[:, 1]
lr_score_test = lr_model.predict_proba(X_test)[:, 1]

lr_thresh, lr_val_f1 = sweep_threshold_f1(y_val, lr_score_val)
print(f"LR best val threshold : {lr_thresh:.6f}  val_F1={lr_val_f1:.4f}")

lr_metrics = evaluate_model("LogisticRegression", y_test, lr_score_test, lr_thresh)
results_log.append(lr_metrics)
print(f"Test metrics          : {lr_metrics}")

print("\nPer-label detection rate on test:")
for cls in sorted(set(label_test)):
    mask = label_test == cls
    rate = (lr_score_test[mask] >= lr_thresh).mean()
    print(f"  {cls:12s}: {mask.sum():6,} samples  detected={rate:.3f}")

In [ ]:
# ── Random Forest ──────────────────────────────────────────────────────────────
print("Training RandomForestClassifier ...")
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

rf_score_val  = rf_model.predict_proba(X_val)[:, 1]
rf_score_test = rf_model.predict_proba(X_test)[:, 1]

rf_thresh, rf_val_f1 = sweep_threshold_f1(y_val, rf_score_val)
print(f"RF best val threshold : {rf_thresh:.6f}  val_F1={rf_val_f1:.4f}")

rf_metrics = evaluate_model("RandomForest", y_test, rf_score_test, rf_thresh)
results_log.append(rf_metrics)
print(f"Test metrics          : {rf_metrics}")

print("\nPer-label detection rate on test:")
for cls in sorted(set(label_test)):
    mask = label_test == cls
    rate = (rf_score_test[mask] >= rf_thresh).mean()
    print(f"  {cls:12s}: {mask.sum():6,} samples  detected={rate:.3f}")

## Phase 7 — Results Comparison

Comparative table: ROC-AUC / F1 / Precision@Recall≥0.9 on the test set.  
Overlaid ROC + PR curves.  Per-label detection rate breakdown for `benign`, `malicious`, `outlier`.

In [ ]:
# ── Comparative metrics table (all 4 models × test set) ───────────────────────
results_df = (
    pd.DataFrame(results_log)
    .set_index("model")
    [["roc_auc", "f1", "p_at_r90"]]
)

print("=== Test Set Metrics ===")
print(results_df.to_string())

# Sanity: all models should beat random (ROC-AUC > 0.5)
failing = results_df[results_df["roc_auc"] <= 0.5]
if len(failing) == 0:
    print("\n✓ All models exceed ROC-AUC > 0.5")
else:
    print(f"\n⚠ Models <= ROC-AUC 0.5: {failing.index.tolist()}")

results_df.to_csv(OUTPUT_DIR / "model_comparison.csv")
print(f"Saved → {OUTPUT_DIR / 'model_comparison.csv'}")
results_df

In [ ]:
# ── Collect all model scores & thresholds ─────────────────────────────────────
model_scores = {
    "IsolationForest"   : if_score_test,
    "Autoencoder"       : ae_mse_test,
    "LogisticRegression": lr_score_test,
    "RandomForest"      : rf_score_test,
}
model_thresholds = {
    "IsolationForest"   : if_thresh,
    "Autoencoder"       : ae_thresh,
    "LogisticRegression": lr_thresh,
    "RandomForest"      : rf_thresh,
}

# ── ROC curves ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for name, scores in model_scores.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Test Set")
ax.legend(fontsize=8)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# ── Precision-Recall curves ───────────────────────────────────────────────────
ax = axes[1]
baseline = float(y_test.mean())
for name, scores in model_scores.items():
    prec, rec, _ = precision_recall_curve(y_test, scores)
    ax.plot(rec, prec, label=name)
ax.axhline(baseline, ls="--", color="grey", lw=0.8, label=f"Baseline (prev={baseline:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–Recall Curves — Test Set")
ax.legend(fontsize=8)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_pr_curves.png", bbox_inches="tight")
plt.show()

# ── Per-label detection rate breakdown ────────────────────────────────────────
print("\n=== Per-label detection rate on test (fraction predicted anomalous) ===")
label_classes = sorted(set(label_test))
breakdown_rows: list[dict] = []

for name in model_scores:
    scores = model_scores[name]
    thresh = model_thresholds[name]
    row = {"model": name}
    for cls in label_classes:
        mask = label_test == cls
        row[cls] = round(float((scores[mask] >= thresh).mean()), 4) if mask.sum() > 0 else float("nan")
    breakdown_rows.append(row)

breakdown_df = pd.DataFrame(breakdown_rows).set_index("model")
print(breakdown_df.to_string())
breakdown_df.to_csv(OUTPUT_DIR / "per_label_detection_rates.csv")
print(f"\nSaved → {OUTPUT_DIR / 'per_label_detection_rates.csv'}")
breakdown_df